<a href="https://colab.research.google.com/github/kimunilaz/Data_Structures-Algorithms_Final_Project/blob/main/Data_Structures%26Algorithms_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Loading the data and building the graph**

This section reads the two input files.



*   Q2_road_network.csv gives 25 roads, each with two junctions and a travel time in minutes.
*   Q2_residential_areas.csv gives 18 residential areas and the junction where each one is located.






The road network is stored as an adjacency list: each junction maps to a list of (neighbour, travel time) pairs. Every road can be travelled in both directions, so each road is added twice, once from each end. The graph is undirected and weighted, with travel time as the weight.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

s_path = "https://raw.githubusercontent.com/kimunilaz/Data_Structures-Algorithms_Final_Project/main/"

def load_csv(filename):
    if os.path.exists(filename):
        return pd.read_csv(filename)          # running locally after cloning
    return pd.read_csv(s_path + filename)     # running in Colab, reads from GitHub

roads = load_csv("Q2_road_network.csv")
areas = load_csv("Q2_residential_areas.csv")

#print(roads.head(5))
#print(areas.head(5))

#building adjacency list

graph = defaultdict(list)

for _, row in roads.iterrows():
    a = row["from"]
    b = row["to"]
    t = row["minutes"]
    graph[a].append((b, t))
    graph[b].append((a, t))   # roads go both ways, so add both directions

junctions = sorted(graph.keys())


#testing
print("Number of junctions:", len(junctions))
print("Number of roads:", sum(len(v) for v in graph.values()) // 2)
print(graph["Curepipe"])



Number of junctions: 18
Number of roads: 25
[('Forest Side', 4), ('Floreal', 6)]


# **Dijkstra's algorithm**

Dijkstra's algorithm finds the shortest travel time from one starting junction to every other junction. It keeps a priority queue of junctions ordered by their best known time. It repeatedly takes the closest unvisited junction and updates its neighbours if a faster path through it is found. It also records each junction's previous junction on the shortest path.


**This is used in two ways:**


*   Fastest route: run from Curepipe, then follow the previous-junction records
back from Pamplemousses to rebuild the actual route, not just its total time.
*   Travel-time table: run once from every junction to get the shortest travel time between every junction and every area. The coverage step uses this table.





Dijkstra works here because all travel times are positive.

In [2]:
import heapq

def dijkstra(graph, start):
    dist = {j: float("inf") for j in graph}
    prev = {j: None for j in graph}
    dist[start] = 0

    pq = [(0, start)]     # (time so far, junction)
    visited = set()
    relaxations = 0

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)

        for v, t in graph[u]:
            new_time = d + t
            if new_time < dist[v]:          # found a faster way to reach v
                dist[v] = new_time
                prev[v] = u
                heapq.heappush(pq, (new_time, v))
                relaxations += 1

    return dist, prev, relaxations


def get_route(prev, start, end):
    route = []
    current = end
    while current is not None:
        route.append(current)
        if current == start:
            break
        current = prev[current]
    route.reverse()
    if route[0] != start:      # end can't be reached from start
        return None
    return route


# Fastest route from Curepipe to Pamplemousses
dist, prev, relax = dijkstra(graph, "Curepipe")
route = get_route(prev, "Curepipe", "Pamplemousses")

print("Fastest route:", " → ".join(route))
print("Travel time:", dist["Pamplemousses"], "minutes")
print("Relaxations:", relax)


# Travel-time table: shortest time from every junction to every other junction
travel_time = {}
for j in junctions:
    d, _, _ = dijkstra(graph, j)
    travel_time[j] = d


# Checks
assert dist["Curepipe"] == 0
assert route[0] == "Curepipe" and route[-1] == "Pamplemousses"

# The route's total from the road times should equal Dijkstra's answer
route_total = 0
for a, b in zip(route, route[1:]):
    route_total += min(t for v, t in graph[a] if v == b)
assert route_total == dist["Pamplemousses"]

# Roads go both ways, so the table should be symmetric
for a in junctions:
    for b in junctions:
        assert travel_time[a][b] == travel_time[b][a]

print("All checks passed")

Fastest route: Curepipe → Forest Side → Phoenix → Quatre Bornes → Rose Hill → Beau Bassin → Port Louis → Terre Rouge → Pamplemousses
Travel time: 59 minutes
Relaxations: 19
All checks passed


# **Coverage**

A stop covers an area if the shortest travel time between the stop's junction and the area's junction is at most nine minutes. Exactly nine minutes counts as covered.


Using the travel-time table from section 2, this section builds a coverage set for each junction: the areas a stop at that junction would cover. Both the greedy method and the exhaustive search use these same sets, so they are compared on identical coverage rules.

In [3]:
COVER_LIMIT = 9   # minutes; exactly 9 counts as covered

# Which junction each area sits at
area_junction = {}
for _, row in areas.iterrows():
    area_junction[row["area"]] = row["junction"]

# Every area junction must exist in the road network
for a, j in area_junction.items():
    assert j in graph, f"Area {a} sits at {j}, which is not in the road network"

# For each junction, the set of areas a stop there would cover
coverage = {}
for j in junctions:
    covered = set()
    for a, aj in area_junction.items():
        if travel_time[j][aj] <= COVER_LIMIT:
            covered.add(a)
    coverage[j] = covered


# Show the results
print(f"{'Junction':<20} {'Areas covered':>13}")
for j in junctions:
    print(f"{j:<20} {len(coverage[j]):>13}")


# Checks
assert len(area_junction) == 18

# A stop at an area's own junction is 0 minutes away, so it must cover that area
for a, aj in area_junction.items():
    assert a in coverage[aj]

print("Coverage checks passed")

KeyError: 'junction'

# **Greedy stop placement**

This is the main method. It chooses six stops one at a time. At each step it adds the junction that covers the most areas not already covered by the stops chosen so far. Once a stop is chosen, it is never removed or changed.

If two or more junctions cover the same number of new areas, the one whose name comes first alphabetically is chosen. This makes the result fixed and repeatable.

Greedy is fast, but it is not guaranteed to find the best set of six. It only looks at the best choice at each step, not at how choices work together.

In [ ]:
NUM_STOPS = 6

def greedy_stops(coverage, junctions, k):
    chosen = []
    covered = set()
    checks = 0          # operation count: how many junctions were evaluated

    for step in range(1, k + 1):
        best = None
        best_gain = -1

        # Going through junctions in alphabetical order and only replacing
        # on a strictly bigger gain means ties go to the earlier name
        for j in sorted(junctions):
            if j in chosen:
                continue
            gain = len(coverage[j] - covered)   # areas this stop would newly cover
            checks += 1
            if gain > best_gain:
                best = j
                best_gain = gain

        chosen.append(best)
        covered |= coverage[best]
        print(f"Stop {step}: {best:<20} new areas: {best_gain:>2}   total covered: {len(covered)}")

    return chosen, covered, checks


greedy_chosen, greedy_covered, greedy_checks = greedy_stops(coverage, junctions, NUM_STOPS)

print("\nGreedy stops:", greedy_chosen)
print("Areas covered:", len(greedy_covered), "out of", len(area_junction))
print("Uncovered areas:", sorted(set(area_junction) - greedy_covered))
print("Junctions evaluated:", greedy_checks)


# Checks
assert len(greedy_chosen) == NUM_STOPS
assert len(set(greedy_chosen)) == NUM_STOPS          # no junction picked twice

union = set()
for j in greedy_chosen:
    union |= coverage[j]
assert union == greedy_covered

print("Greedy checks passed")

# **Exhaustive search (baseline)**

This is the baseline the greedy method is compared against. It tries every possible set of six junctions out of 18, which is C(18, 6) = 18,564 combinations. For each set it counts how many areas are covered, and it keeps the best.

Because it checks every option, the result is the true maximum coverage. The gap between this number and the greedy result shows how far greedy fell short on this network.

A counter records how many combinations were checked, alongside the measured running time.